# Utilities

> Utility Functions

In [ ]:
#| default_exp utils

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| export

# =================================
# Standard library
# =================================
import yaml
from random import randint, random as rand, choice
from dataclasses import dataclass, field
from pathlib import Path, PurePath
from types import SimpleNamespace
from typing import Any, List, Optional, Union
import inspect

# =================================
# Imaging
# =================================
from skimage import util

# =================================
# PyTorch
# =================================
from torch import (
    Tensor as torchTensor,
    squeeze as torchsqueeze,
    max as torchmax,
    from_numpy as torch_from_numpy,
    device as torch_device,
)

from torch.cuda import is_available as is_cuda_available

# =================================
# MONAI
# =================================
from monai.data import MetaTensor
from monai.utils import set_determinism
from collections.abc import Callable, Iterable, Sequence, MutableSequence
from monai.config import PathLike
from monai.utils.misc import ensure_tuple, ensure_tuple_rep

# =================================
# fastai
# =================================
from fastai.data.all import delegates, hasattrs, L
from fastai.vision.all import store_attr, BypassNewMeta, DisplayedTransform

# =================================
# fastcore
# =================================
from fastcore.script import risinstance

# =================================
# Multiple dispatch
# =================================
from plum import dispatch as typedispatch

In [ ]:
#| export
delegates = delegates
hasattrs = hasattrs
List = List
L = L
Any = Any
store_attr = store_attr
BypassNewMeta = BypassNewMeta
DisplayedTransform = DisplayedTransform

dataclass = dataclass
field = field

risinstance = risinstance

typedispatch = typedispatch
MetaTensor = MetaTensor
set_determinism = set_determinism

Callable = Callable
Iterable = Iterable
Sequence = Sequence
MutableSequence = MutableSequence
Optional = Optional
Union = Union

Path = Path
PurePath = PurePath
PathLike = PathLike

ensure_tuple = ensure_tuple
ensure_tuple_rep = ensure_tuple_rep

torchTensor = torchTensor
torch_from_numpy = torch_from_numpy
torch_device = torch_device
torchsqueeze = torchsqueeze
torchmax = torchmax

is_cuda_available = is_cuda_available 

In [ ]:
show_doc(delegates)
# show_doc(hasattrs)
# show_doc(List)
# show_doc(L)
# show_doc(Any)
# show_doc(store_attr)
# show_doc(risinstance)
# show_doc(typedispatch)
# show_doc(MetaTensor)
# show_doc(set_determinism)
# show_doc(Callable)
# show_doc(Iterable)
# show_doc(Sequence)
# show_doc(PathLike)
# show_doc(torchTensor)
# show_doc(torch_from_numpy)
# show_doc(torch_device)

---

[source](https://github.com/AnswerDotAI/fastcore/blob/main/fastcore/meta.py#LNone){target="_blank" style="float:right; font-size:smaller"}

### delegates

```python

def delegates(
    to:function=None, # Delegatee
    keep:bool=False, # Keep `kwargs` in decorated function?
    but:list=None, # Exclude these parameters from signature
    sort_args:bool=False, # Sort arguments alphabetically, doesn't work with call_parse
):


```

*Decorator: replace `**kwargs` in signature with params from `to`*

The utils module contains helper functions and classes to facilitate data manipulation, model setup, and training. These utilities add flexibility and convenience, supporting rapid experimentation and efficient data handling.


In [ ]:
#| export
def add_method(cls):
    def decorator(func):
        setattr(cls, func.__name__, func)
        return func
    return decorator

In [ ]:
#| export
def attributesFromDict(d):
    """
    The `attributesFromDict` function simplifies the conversion of dictionary keys and values into object attributes, allowing dynamic attribute creation for configuration objects. This utility is handy for initializing model or dataset configurations directly from dictionaries, improving code readability and maintainability.
    """
    self = d.pop('self')
    for n, v in d.items():
        setattr(self, n, v)

In [ ]:
#| export
def get_device():
    """
    The `get_device` function is used to detect if the device the code is executed in has got a CUDA-enabled GPU available. 
    If it doesn’t, it returns CPU. 
    """ 
    return torch_device("cuda" if is_cuda_available() else "cpu")

In [ ]:
#| export
def img2float(image, force_copy=False):
    """
    The `img2float` function turns an image into float representation.
    """
    return util.img_as_float(image, force_copy=force_copy)

In [ ]:
#| export
def img2Tensor(image):
    """
    The `img2Tensor` function turns an image into tensor representation after turning it first into float representation. 
    """
    return torchTensor(img2float(image))

In [ ]:
#| export
def route_kwargs(func, kwargs):
    """
    Filter a dictionary of kwargs to only include those accepted by `func`.

    Handles:
      - Explicit parameters
      - Functions with **kwargs (all extra keys are allowed)
    """
    sig = inspect.signature(func)
    accepts_kwargs = any(
        p.kind == inspect.Parameter.VAR_KEYWORD for p in sig.parameters.values()
    )

    if accepts_kwargs:
        # If func accepts **kwargs, pass everything
        return kwargs.copy()
    else:
        # Otherwise, filter to matching parameters only
        return {k: v for k, v in kwargs.items() if k in sig.parameters}

Routes kwargs only to functions that accept them.

In [ ]:
#| export
def read_yaml(yaml_path):
    "Reads a YAML file and returns its contents as a dictionary"
    with open(yaml_path, 'r') as file:
        config = yaml.safe_load(file)
    return config 

In [ ]:
def read_args_from_yaml(yaml_path):
    """Reads arguments from a YAML file and converts them into a namespace."""
    config = read_yaml(yaml_path)
    if config is None:
        config = {}

    def _convert(value):
        if isinstance(value, dict):
            return SimpleNamespace(**{k: _convert(v) for k, v in value.items()})
        if isinstance(value, list):
            return [_convert(v) for v in value]
        return None if value == "None" else value

    return _convert(config)

In [ ]:
#| export
def dictlist_to_funclist(transform_dicts):
    transforms = []
    for trans in transform_dicts:
        if isinstance(trans, str):  
            transform_obj = globals().get(trans)
            transforms.append(transform_obj)
        else: 
            name, params = next(iter(trans.items()))
            transform_obj = globals().get(name) or eval(name) 
            transforms.append(transform_obj(**params))

    return transforms

In [ ]:
#| export

@dataclass
class TargetedTransform:
    """
    Wrapper for a transform that specifies which input(s) it should be applied to.

    This allows fine-grained control when working with paired data such as
    (X, y), stereo images, or multi-modal inputs.

    Parameters
    ----------
    transform : callable
        The transform to apply. Must implement an `encodes()` method
        if used within a Transform pipeline.

    targets : tuple of str, default ("both",)
        Specifies where the transform should be applied.
        Supported values:
            - ("X",)      : apply only to the first element
            - ("y",)      : apply only to the second element
            - ("both",)   : apply to both elements

    Examples
    --------
    Apply to both inputs (default):

        TargetedTransform(RandomFlip())

    Apply only to X:

        TargetedTransform(RandomBrightness(), targets=("X",))

    Apply only to y:

        TargetedTransform(RemapMask(), targets=("y",))
    """
    transform: callable
    targets: tuple = ("both",)   # ("X",), ("y",), ("both",)


In [ ]:
#| export
def apply_transforms(image, transforms):
    """Apply a list of transformations, ensuring at least one is applied.
    
    Supports:
        - plain transforms (applied to both images if tuple)
        - TargetedTransform(transform, targets=...)
    """
    if not transforms:
        return image

    # Normalize transforms into TargetedTransform objects
    normalized = []
    for t in transforms:
        if isinstance(t, TargetedTransform):
            normalized.append(t)
        else:
            # Treat normal transforms as applied to both
            normalized.append(TargetedTransform(transform=t, targets=("both",)))

    # Randomly select transforms based on probability p if present
    applied = [
        spec for spec in normalized
        if not hasattr(spec.transform, "p") or rand() < spec.transform.p
    ]

    # Ensure at least one transform is applied
    if not applied:
        applied.append(choice(normalized))

    def apply_transform_to_image(img, transform):
        return transform.encodes(img)

    # ---- Single image case ----
    if not isinstance(image, tuple):
        for spec in applied:
            image = apply_transform_to_image(image, spec.transform)
        return image

    # ---- Tuple case ----
    image1, image2 = image

    for spec in applied:
        t = spec.transform
        targets = spec.targets

        if "both" in targets or "X" in targets:
            image1 = apply_transform_to_image(image1, t)

        if "both" in targets or "y" in targets:
            image2 = apply_transform_to_image(image2, t)

    return image1, image2

In [ ]:
# If we pass an empty list of transforms, it should return the input unchanged
test_eq(apply_transforms([1, 2], []), [1, 2]) 

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()